# L23 · AI Agent：会自己规划的助手

**学习目标**
- 理解「Agent（智能体）」：能自主规划、调用工具、直到完成任务
- 理解 ReAct 范式：推理(Thought) → 行动(Act) → 观察(Observe) 循环
- 亲手实现一个「自动分解任务并执行」的迷你 Agent（离线）

**前置依赖**：L21（函数调用）、L22（RAG）、L06（面向对象）  
**预计时长**：50 分钟  
**技术栈**：纯 Python（离线模拟规划与执行）

---

## 概念讲解：Agent = 会自己想办法的实习生

普通 AI 是一问一答。Agent 像一个**会自己干的实习生**：
你给个目标「帮我订明天北京出差的差旅」，它会：
1. **想**（Thought）：需要先查天气、查机票、订酒店
2. **做**（Act）：调用查天气工具 → 拿到结果
3. **看**（Observe）：根据结果决定下一步
4. 循环直到任务完成

这就是 **ReAct（推理+行动）** 范式，当今最强 Agent 的底层逻辑。

## 第一步：给 Agent 装备几个工具

In [ ]:
import random
def tool_weather(city): return f"{city} 明天 {random.choice(['晴','多云','雨'])}，22°C"
def tool_flight(src, dst): return f"{src}→{dst} 最便宜 ¥{random.randint(500,1500)}"
def tool_hotel(city): return f"{city} 商务酒店 ¥{random.randint(300,800)}/晚"
def tool_calendar(text): return f"已记入日历：{text}"

TOOLS = {"weather": tool_weather, "flight": tool_flight, "hotel": tool_hotel, "calendar": tool_calendar}
print("Agent 已装备工具：", list(TOOLS))

## 第二步：一个「规划器」—— 把目标拆成步骤

In [ ]:
def plan(task):
    """极简规划：依据关键词拆出子任务（真 LLM 由模型动态生成）"""
    steps = []
    if "天气" in task or "出差" in task:
        steps.append(("weather", ["北京"]))
    if "机票" in task or "出差" in task:
        steps.append(("flight", ["上海", "北京"]))
    if "酒店" in task or "出差" in task:
        steps.append(("hotel", ["北京"]))
    steps.append(("calendar", [task]))
    return steps

print("任务拆解：", plan("帮我安排北京出差"))

## 第三步：ReAct 循环执行

In [ ]:
def run_agent(task):
    log = []
    for tool_name, args in plan(task):
        log.append(f"  💭 Thought: 需要调用 {tool_name}")
        result = TOOLS[tool_name](*args)
        log.append(f"  🛠️  Act: {tool_name}{args} → {result}")
        log.append(f"  👁️  Observe: 获得信息，继续下一步\n")
    log.append("  ✅ 任务完成！")
    return "\n".join(log)

print(run_agent("帮我安排北京出差"))

# 🎯 AHA 顿悟单元格：你的「自主差旅 Agent」

运行下面代码。你会看到一个 Agent **自己把「安排出差」拆成 天气→机票→酒店→日历 四步**，
一步步调用工具、记录观察，最后汇报「任务完成」。改任务文本试试不同拆解。

> 这就是今天最前沿的 AI Agent：不是你一步步指挥，而是它自己规划执行。
> 你刚写的 `plan + run_agent` 循环，和 OpenAI/Anthropic 的 Agent 框架是同一个灵魂。

In [ ]:
# ===== 运行我！看 Agent 自主规划执行 =====
tasks = [
    "帮我安排北京出差",
    "查一下北京天气并记入日历",
    "订北京酒店",
]
for t in tasks:
    print(f"🎯 用户目标：{t}")
    print(run_agent(t))
    print("=" * 50 + "\n")
print("  🚀 你的 Agent 已经能自主干活了！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：ReAct 循环心智；规划器「为何能拆步骤」。需明说真 Agent 由 LLM 动态生成 Thought/Act，本课用规则 `plan()` 模拟其核心结构。  
**易错点**：工具签名匹配（args 数量）；无限循环风险（真 Agent 需 max_steps 上限，本方案 steps 有限）。  
**AHA 机制**：自主多步规划+工具链，强「AI 自己干活」惊喜，且零依赖。  
**衔接**：L24 微调（让 Agent 的「大脑」更聪明）；L38 综合项目（多 Agent 协作）。  
**真 LLM 衔接**：注明生产用 LLM 输出 JSON `{thought, action, args}` 驱动循环，并加 `max_iter` 安全阀。  
**安全红线**：强调 Agent 执行真实动作（发邮件/下单）必须有人类确认环节。

# 📚 作业 / 下一步

1. 给 `plan` 加一个「订餐」分支和 `tool_order_food` 工具。
2. 思考：如果工具失败怎么办？（引出 Agent 的「重试/反思」机制，L25+ 工程化）
3. 下一课 **L24 微调入门：教 AI 说你的语言** —— 用你自己的数据，轻轻「调教」一个模型。